In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # used to do transformation on image

# transforms.compose used to chain transform for a image

# image[0-255] --> Scale(0 , 1) --> Normalize(-1 , 1)

transform = transforms.Compose([
    transforms.ToTensor(), # convert all images to pyTorch tensors + scale all of them
    transforms.Normalize((0.5 , 0.5 , 0.5) , (0.5 , 0.5 , 0.5)) # standard deviation and means are passed
])

trainset = CIFAR10(root = "./data" , train = True , download = True , transform = transform)
testset = CIFAR10(root = "./data" , train = False , download = True , transform = transform)


# TorchVision
* **Datasets**
    * CIFAR10
    * MNIST
* **Pretrained CNNs**
* **Utilities for image transformation**

In [3]:
trainLoader = DataLoader(trainset , batch_size = 64 , shuffle = True )
testLoader = DataLoader(testset , batch_size = 64)

### Build CNN Model

In [6]:
# CNN --> (convo + ReLU) -- maxPool --> (convo + ReLU) -- maxPool --> (convo + ReLU) -- maxPool --> flattening --> FC Layer
# Kernel = 3 * 3 , padding = 1 & maxPool kernel = 2 * 2 , stride = 2
# multi-class classification --> CrossEntropyLoss --> In pyTorch crossEntropyLoss uses in-built softmax

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN , self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3 , 32 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2), # kernel_size = 2 , stride = 2

            nn.Conv2d(32 , 64 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(64 , 128 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),

            nn.Linear(256 , 10)
        )

    def forward(self , x):
        x = self.conv_layers(x)
        x = x.view(x.size(0) , -1) # flattening
        x = self.fc_layers(x)

        return x

In [9]:
model = CNN()

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training CNN

In [11]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images , labels in trainLoader:
        optimizer.zero_grad()
        
        output = model.forward(images)
        loss = criterion(output , labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainLoader)}")

epoch = 1/10 & loss = 1.377822090445272
epoch = 2/10 & loss = 0.944263802007641
epoch = 3/10 & loss = 0.7619212891744531
epoch = 4/10 & loss = 0.6269206519779342
epoch = 5/10 & loss = 0.5278006478610551
epoch = 6/10 & loss = 0.4336630939065343
epoch = 7/10 & loss = 0.3524130440181326
epoch = 8/10 & loss = 0.2760303753625859
epoch = 9/10 & loss = 0.21002562350267187
epoch = 10/10 & loss = 0.17033808366121614


In [13]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images , labels in testLoader:
        outputs = model.forward(images)
        _ , predicted = torch.max(outputs , 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.77000000000001
